# Notebook 7 – Encoding Categorical Variables
## What is Categorical Encoding?

Machine learning models work with numbers, not text. Categorical encoding
is how we convert columns like `Department` or `Education` into numeric
form, without accidentally inventing a false relationship between categories
that never existed in the real world.

That last part is where most mistakes happen. Let's go through it using
our own columns.

In [10]:
import pandas as pd

df = pd.read_csv("hr_employee_attrition_raw.csv")
for col in ["Gender", "Department", "JobRole", "Education", "OverTime", "Attrition"]:
    print(col, "->", df[col].nunique(), "unique values:", df[col].unique()[:6])

Gender -> 6 unique values: <ArrowStringArray>
['Female', 'Male', nan, 'M', 'FEMALE', 'male']
Length: 6, dtype: str
Department -> 9 unique values: <ArrowStringArray>
['Finance', 'sales', 'HR', 'Support', 'Sales', 'Engineering']
Length: 6, dtype: str
JobRole -> 13 unique values: <ArrowStringArray>
[       'Accountant',     'Brand Manager',        'HR Manager',
      'Support Lead', 'Support Executive',     'Sales Manager']
Length: 6, dtype: str
Education -> 4 unique values: <ArrowStringArray>
['Bachelor's', 'High School', nan, 'Master's', 'PhD']
Length: 5, dtype: str
OverTime -> 6 unique values: <ArrowStringArray>
['No', 'Yes', 'yes', 'Y', 'no', 'N']
Length: 6, dtype: str
Attrition -> 2 unique values: <ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str


## Nominal vs Ordinal vs Binary Variables

Before picking a technique, classify the column. This decision drives
everything else.

* **Nominal** = categories with no natural order. `Department` (Finance,
  Sales, HR) is nominal, Finance isn't "more" or "less" than Sales.
* **Ordinal** = categories with a real, meaningful order. `Education`
  (High School < Bachelor's < Master's < PhD) is ordinal, there's a clear
  ranking.
* **Binary** = only two possible categories. `Attrition` (Yes/No),
  `OverTime` (Yes/No), and `Gender` (once cleaned to two values) are binary.

Getting this classification wrong is the root cause of most encoding
mistakes below.

## Label Encoding

**What it does:** assigns each category an arbitrary integer (0, 1, 2, ...).

**When to use it:** only for **ordinal** variables where the order is real,
or for tree-based models (Random Forest, XGBoost) that don't assume numeric
distance means anything.

**Risk if used wrong:** applying label encoding to a nominal column like
`Department` implies `Sales (2) > Finance (0)`, a relationship that doesn't
exist. A linear or distance-based model (like linear regression or KNN)
will treat that gap as real and learn a false pattern.

In [11]:
from sklearn.preprocessing import LabelEncoder

# WRONG use: Department has no order, label encoding fakes one
le = LabelEncoder()
df["Department_label_wrong_example"] = le.fit_transform(df["Department"].fillna("Unknown"))
print(dict(zip(le.classes_, le.transform(le.classes_))))

{'Engineering': np.int64(0), 'Finance': np.int64(1), 'HR': np.int64(2), 'Marketing': np.int64(3), 'SALES': np.int64(4), 'Sales': np.int64(5), 'Sales ': np.int64(6), 'Support': np.int64(7), 'sales': np.int64(8)}


## Ordinal Encoding

**What it does:** same as label encoding, but the integers are assigned
**deliberately**, in the correct real-world order.

**When to use it:** `Education` is the perfect fit here. High School should
map to a lower number than PhD, on purpose, not by whatever order pandas
happens to sort alphabetically.

**Risk if skipped:** if you let `LabelEncoder` auto-assign values instead
of manually ordering them, you might end up with `PhD = 0` and
`High School = 3`, silently reversing the real ranking.

In [12]:
education_order = {"High School": 1, "Bachelor's": 2, "Master's": 3, "PhD": 4}
df["Education_ordinal"] = df["Education"].map(education_order)
df[["Education", "Education_ordinal"]].drop_duplicates()

,Education,Education_ordinal
0,Bachelor's,2.0
1,High School,1.0
2,NaN,NaN
6,Master's,3.0
18,PhD,4.0


## Binary Variables

**What it does:** maps a two-category column to 0/1 directly, no separate
technique needed.

**When to use it:** `OverTime` and `Attrition` are naturally binary, once
cleaned. Right now `OverTime` has messy values (`Yes`, `yes`, `Y`, `No`,
`no`, `N`), so cleaning has to happen before encoding, not after.

**Risk if skipped:** encoding `Yes`, `yes`, and `Y` as three different
categories (instead of recognizing them as one) fragments the column and
weakens any pattern the model could have learned from it.

## Binary Variables

**What it does:** maps a two-category column to 0/1 directly, no separate
technique needed.

**When to use it:** `OverTime` and `Attrition` are naturally binary, once
cleaned. Right now `OverTime` has messy values (`Yes`, `yes`, `Y`, `No`,
`no`, `N`), so cleaning has to happen before encoding, not after.

**Risk if skipped:** encoding `Yes`, `yes`, and `Y` as three different
categories (instead of recognizing them as one) fragments the column and
weakens any pattern the model could have learned from it.

In [13]:
# Clean Department first (case + whitespace), THEN one-hot encode
df["Department_clean"] = df["Department"].str.strip().str.title()
department_dummies = pd.get_dummies(df["Department_clean"], prefix="Dept", drop_first=True)
department_dummies.head()

,Dept_Finance,Dept_Hr,Dept_Marketing,Dept_Sales,Dept_Support
0,True,False,False,False,False
1,True,False,False,False,False
2,False,False,False,True,False
3,False,True,False,False,False
4,False,False,False,False,True


## Frequency Encoding

**What it does:** replaces each category with how often it appears in the
dataset.

**When to use it:** useful for medium/high-cardinality nominal columns like
`JobRole`, where one-hot encoding would create too many sparse columns, but
you still want to capture some signal from the category.

**Risk:** two genuinely different categories can end up with the same
encoded value purely by coincidence, if they happen to appear the same
number of times. The model then can't tell them apart, even though they
may behave very differently.

In [18]:
freq_map = df["JobRole"].value_counts(normalize=True)
df["JobRole_freq_encoded"] = df["JobRole"].map(freq_map)
df[["JobRole", "JobRole_freq_encoded"]].drop_duplicates().head()

,JobRole,JobRole_freq_encoded
0,Accountant,0.088618
2,Brand Manager,0.081301
3,HR Manager,0.086992
4,Support Lead,0.086992
5,Support Executive,0.080488


## Target Encoding

**What it does:** replaces each category with the average value of the
target variable for that category (e.g., average attrition rate per
`JobRole`).

**When to use it:** high-cardinality nominal columns in a supervised
learning setup, where you specifically want the encoding to carry
predictive signal about the target.

**Risk (the big one): data leakage.** If you compute the average attrition
rate per `JobRole` using the *entire* dataset (including the rows you'll
later test on), the model indirectly "sees" the target during training.
This inflates performance during testing and quietly falls apart on new,
unseen data. Target encoding must be done using cross-validation folds or
only on the training split, never the full dataset.

In [22]:
role_attrition_rate = train_df.groupby("JobRole")["Attrition_binary"].mean()
train_df["JobRole_target_encoded"] = train_df["JobRole"].map(role_attrition_rate)
test_df["JobRole_target_encoded"] = test_df["JobRole"].map(role_attrition_rate)

train_df[["JobRole", "JobRole_target_encoded"]].head()

,JobRole,JobRole_target_encoded
753,Sales Executive,0.123077
1068,Analyst,0.183908
1182,HR Manager,0.225000
425,Marketing Executive,0.150000
644,Brand Manager,0.278481


## Handling Unknown Categories

**The problem:** what happens when the model sees a category during
prediction that it never saw during training? For example, a new
`JobRole` like "Data Scientist" gets added to the company after the model
was trained.

**How to handle it:**
* One-hot encoding: the new category simply gets all zeros across the
  existing dummy columns (it won't crash, but it also won't be recognized).
* Label/ordinal encoding: needs an explicit fallback value (e.g., -1 or
  "Unknown") set in advance, or the pipeline will throw an error.
* Frequency/target encoding: map unseen categories to a sensible default,
  like the overall average, not zero.

**Risk if not planned for:** a pipeline that works fine in testing can
crash in production the first time a genuinely new category appears.

In [24]:
print(df[["JobRole", "JobRole_freq_safe"]].head())

         JobRole  JobRole_freq_safe
0     Accountant           0.088618
1     Accountant           0.088618
2  Brand Manager           0.081301
3     HR Manager           0.086992
4   Support Lead           0.086992


## High-Cardinality Categories

`EmployeeID` has over a thousand unique values, essentially one per row.
This is the extreme case worth calling out on its own.

* One-hot encoding it would create over a thousand new columns, almost
  entirely 0s, mostly noise.
* Label encoding it would imply a fake numeric order between employees.
* Frequency encoding it would give every row nearly the same value (since
  each ID appears once), providing no useful signal at all.

**The real answer:** don't encode identifier columns like `EmployeeID` as
a feature at all. It's a key for joining/tracking data, not a predictive
variable.

In [25]:
print("EmployeeID" in model_features.columns)   # should print False
model_features.head()

False


,Age,Gender,Department,JobRole,Education,MonthlyIncome,YearsAtCompany,JobSatisfaction,PerformanceRating,DistanceFromHomeKM,...,JoinDate,Attrition,EmployeeCountFlag,RandomSurveyCode,Department_label_wrong_example,Education_ordinal,Department_clean,JobRole_freq_encoded,Attrition_binary,JobRole_freq_safe
0,23,Female,Finance,Accountant,Bachelor's,2227.56,8,3.0,1,1.8,...,2019-07-23,No,1,448909,1,2.0,Finance,0.088618,0,0.088618
1,29,Male,Finance,Accountant,High School,5558.6,15,1.0,3,5.7,...,2022-02-16,No,1,724794,1,1.0,Finance,0.088618,0,0.088618
2,45,Female,sales,Brand Manager,NaN,6167.63,3,4.0,1,16.7,...,2016-07-16,Yes,1,571964,8,NaN,Sales,0.081301,1,0.081301
3,35,Male,HR,HR Manager,Bachelor's,5492.97,0,1.0,2,0.8,...,2024-08-19,No,1,397616,2,2.0,Hr,0.086992,0,0.086992
4,34,Male,Support,Support Lead,Bachelor's,5433.06,1,NaN,3,9.6,...,2015-03-23,No,1,441126,7,2.0,Support,0.086992,0,0.086992
